<a href="https://colab.research.google.com/github/azizajamjoom/bsan6200-assignment5/blob/main/assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 Option C: LM Evaluation Report
# British Airways Review Sentiment Classification

---

## Task Definition

**Task:** Sentiment classification of British Airways passenger reviews  
**Input:** Free-text passenger review (one sentence to several paragraphs)  
**Output:** Exactly one of three labels — `Positive`, `Negative`, or `Mixed`

### Label Definitions
| Label | Meaning |
|-------|---------|
| `Positive` | Overall tone is satisfied or complimentary, even if minor issues are noted |
| `Negative` | Overall tone is dissatisfied, frustrated, or critical |
| `Mixed` | Clearly praises some aspects AND criticizes others with roughly equal weight |

### Why This Task Matters for Business
Airlines receive thousands of online reviews monthly. Manually reading each one is expensive and slow. An automated sentiment classifier enables British Airways to:
- Instantly flag negative reviews for urgent customer recovery responses
- Track sentiment trends across routes, cabin classes, and staff teams
- Power real-time dashboards for operations leadership
- Reduce analyst triage time by automating clear-cut cases

### Success Criteria
- **Accuracy >= 75%** across all 30 test examples
- **Per-class F1 >= 0.65** for each of the three sentiment labels
- **Format compliance >= 90%** — model returns exactly one valid label
- **Average latency < 8 seconds** per API call
- **Full reproducibility** — seeds fixed, all results saved to CSV

---
## Step 1 — Install Libraries and Configure API Keys

In [1]:
!pip install -q huggingface_hub google-generativeai pandas scikit-learn

In [3]:
import pandas as pd
import time
import os
import re
from sklearn.metrics import accuracy_score, f1_score, classification_report

# API Keys — add via Colab left sidebar > key icon > Secrets
# HF_TOKEN  : get free at hf.co/settings/tokens
# GEMINI_KEY: get free at aistudio.google.com
try:
    from google.colab import userdata
    HF_TOKEN   = userdata.get('HF_TOKEN')
    GEMINI_KEY = userdata.get('GEMINI_KEY')
    print('Keys loaded from Colab Secrets')
except Exception:
    HF_TOKEN   = 'PASTE_YOUR_HF_TOKEN_HERE'
    GEMINI_KEY = 'PASTE_YOUR_GEMINI_KEY_HERE'
    print('WARNING: using hardcoded keys - switch to Secrets before submitting')

import google.generativeai as genai
genai.configure(api_key=GEMINI_KEY)

from huggingface_hub import InferenceClient
hf_client = InferenceClient(token=HF_TOKEN)

VALID_LABELS = ['Positive', 'Negative', 'Mixed']
print('Setup complete.')

Keys loaded from Colab Secrets
Setup complete.


---
## Step 2 — Load Raw Data

In [4]:
from google.colab import files
print('Select reviews_data1.csv when the picker opens...')
uploaded = files.upload()

Select reviews_data1.csv when the picker opens...


Saving reviews_data1.csv to reviews_data1 (1).csv


In [5]:
df_raw = pd.read_csv('reviews_data1.csv')
df_raw['review_len'] = df_raw['Reviews'].str.len()

print(f'Dataset: {df_raw.shape[0]:,} reviews | Columns: {df_raw.columns.tolist()}')
print(f"\nRecommended distribution:")
print(df_raw['Recommended'].value_counts())
print(f"\nReview length distribution:")
print(df_raw['review_len'].describe().round(0))
df_raw.head(3)

Dataset: 3,427 reviews | Columns: ['Verified', 'Reviews', 'Recommended', 'review_len']

Recommended distribution:
Recommended
no     1990
yes    1437
Name: count, dtype: int64

Review length distribution:
count    3427.0
mean      889.0
std       568.0
min        83.0
25%       486.0
50%       747.0
75%      1121.0
max      3537.0
Name: review_len, dtype: float64


,Verified,Reviews,Recommended,review_len
0,yes,i was flying to warsaw for one day of mee...,no,1373
1,yes,"booked a ba holiday to marrakech, after p...",yes,473
2,yes,extremely sub-par service. highlights: no ...,no,784


---
## Step 3 — Build Test Set (30 Examples, 3 Complexity Tiers)

The 30 examples are split across three complexity tiers:

| Tier | Count | Description |
|------|-------|-------------|
| **Easy** | 20 | Clear-cut sentiment — unambiguously positive or negative reviews |
| **Hard** | 5 | Mix of praise and criticism, hedged language, or nuanced opinions |
| **Really Difficult** | 5 | Very short (insufficient context), very long multi-issue reviews |



In [6]:
import random
random.seed(42)

# EASY (20): clear positive and clear negative, medium-length reviews
# Recommended='yes' strongly correlates with positive; 'no' with negative
easy_pos = df_raw[
    (df_raw['Recommended'] == 'yes') &
    (df_raw['review_len'].between(250, 800))
].sample(10, random_state=42).copy()

easy_neg = df_raw[
    (df_raw['Recommended'] == 'no') &
    (df_raw['review_len'].between(250, 800))
].sample(10, random_state=42).copy()

easy_pos['difficulty'] = 'easy'
easy_neg['difficulty'] = 'easy'
used = set(easy_pos.index) | set(easy_neg.index)

# HARD (5): longer verified reviews with likely mixed signals
# Verified users tend to write more detailed, nuanced, mixed-tone reviews
hard = df_raw[
    (df_raw['Verified'] == 'yes') &
    (df_raw['review_len'].between(800, 1600)) &
    (~df_raw.index.isin(used))
].sample(5, random_state=7).copy()

hard['difficulty'] = 'hard'
used |= set(hard.index)

# REALLY DIFFICULT (5): very short OR very long reviews
# Short: barely enough text to judge sentiment
# Long: multiple topics spanning an entire multi-leg journey
rd_short = df_raw[
    (df_raw['review_len'] < 150) &
    (~df_raw.index.isin(used))
].sample(2, random_state=13).copy()

rd_long = df_raw[
    (df_raw['review_len'] > 1800) &
    (~df_raw.index.isin(used))
].sample(3, random_state=13).copy()

rd_short['difficulty'] = 'really_difficult'
rd_long['difficulty']  = 'really_difficult'

# Combine all tiers
test_df = pd.concat([easy_pos, easy_neg, hard, rd_short, rd_long]).reset_index(drop=True)
test_df['id'] = range(1, len(test_df) + 1)
test_df['ground_truth_label'] = ''  # YOU FILL THIS IN

test_set = test_df[['id', 'Reviews', 'Verified', 'Recommended',
                     'review_len', 'difficulty', 'ground_truth_label']].copy()
test_set.rename(columns={'Reviews': 'review_text'}, inplace=True)
test_set.to_csv('test_set.csv', index=False)

print(f'Test set saved: {len(test_set)} examples')
print(f"\nDifficulty breakdown:")
print(test_set['difficulty'].value_counts())
print(f"\nRecommended per tier (use as labeling guide, not a substitute for reading):")
print(test_set.groupby(['difficulty','Recommended']).size())
test_set[['id','difficulty','review_len','review_text']].head(10)

Test set saved: 30 examples

Difficulty breakdown:
difficulty
easy                20
hard                 5
really_difficult     5
Name: count, dtype: int64

Recommended per tier (use as labeling guide, not a substitute for reading):
difficulty        Recommended
easy              no             10
                  yes            10
hard              no              5
really_difficult  no              3
                  yes             2
dtype: int64


,id,difficulty,review_len,review_text
0,1,easy,666,gatwick to venice with british airways. the pl...
1,2,easy,524,i flew from london to milan linate on 27th may...
2,3,easy,264,"shout out to the help desk at heathrow, we ..."
3,4,easy,453,british airways short-haul economy product is ...
4,5,easy,638,mexico city to barcelona via london heathr...
5,6,easy,314,verified review london heathrow to biarri...
6,7,easy,496,"outbound flight to athens was on time, but ret..."
7,8,easy,637,verified review london heathrow to miami ...
8,9,easy,447,heathrow to cape town with british airways. i ...
9,10,easy,421,"very nice and helpful staff in terminal 5, ver..."


In [7]:
files.download('test_set.csv')
print('test_set.csv downloaded.')
print()
print('LABELING GUIDE:')
print('  Positive       - reader is happy/satisfied; would recommend')
print('  Negative       - reader is frustrated/dissatisfied; would not recommend')
print('  Mixed          - genuinely split: praises X, criticizes Y with equal weight')
print()
print('TIP: The Recommended column is a signal but READ the text - some no-recommend')
print('     reviews are calm and factual (Mixed), not clearly Negative.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

test_set.csv downloaded.

LABELING GUIDE:
  Positive       - reader is happy/satisfied; would recommend
  Negative       - reader is frustrated/dissatisfied; would not recommend
  Mixed          - genuinely split: praises X, criticizes Y with equal weight

TIP: The Recommended column is a signal but READ the text - some no-recommend
     reviews are calm and factual (Mixed), not clearly Negative.


---
## Step 4 — Load and Validate Labeled Test Set

In [13]:
print('Upload your hand-labeled test_set_labeled.csv...')
uploaded2 = files.upload()

Upload your hand-labeled test_set_labeled.csv...


Saving Untitled spreadsheet - test_set.csv to Untitled spreadsheet - test_set.csv


In [14]:
import os

# Rename file
os.rename('Untitled spreadsheet - test_set.csv', 'test_set_labeled.csv')

# Load and fix capitalization
test_set = pd.read_csv('test_set_labeled.csv')
test_set['ground_truth_label'] = test_set['ground_truth_label'].str.strip().str.capitalize()

# Validate
assert len(test_set) >= 30, f'Need at least 30 examples, got {len(test_set)}'

invalid = test_set[~test_set['ground_truth_label'].isin(VALID_LABELS)]
assert len(invalid) == 0, f'Invalid labels:\n{invalid[["id","ground_truth_label"]]}'

print(f'Test set validated: {len(test_set)} examples, all labels valid')
print(f"\nLabel distribution:")
print(test_set['ground_truth_label'].value_counts())
print(f"\nDifficulty distribution:")
print(test_set['difficulty'].value_counts())

Test set validated: 30 examples, all labels valid

Label distribution:
ground_truth_label
Negative    15
Positive    11
Mixed        4
Name: count, dtype: int64

Difficulty distribution:
difficulty
easy                20
hard                 5
really_difficult     5
Name: count, dtype: int64


---
## Step 5 — Prompt Templates (3 Strategies per Model)

All templates documented with rationale explaining what each is expected to improve and why.

In [15]:
def get_prompt(strategy: str, review_text: str) -> str:
    """
    Return the full prompt for a given strategy.

    STRATEGY 1 — zero_shot
    Provides only the task description and label list, no examples.
    Rationale: Establishes a performance baseline using only pretrained knowledge.
    Expected to handle clear positive/negative well but struggle with Mixed
    cases where the label boundary is subtle. Format compliance may also be
    lower without examples anchoring the expected single-word output format.

    STRATEGY 2 — few_shot
    Adds 3 labeled examples (one per class) before the query.
    Rationale: Concrete examples anchor the model to the single-word output
    format and explicitly demonstrate the Mixed boundary — the hardest class
    to distinguish. Expected to outperform zero-shot on hard and really
    difficult examples and raise format compliance significantly.

    STRATEGY 3 — cot (Chain-of-Thought)
    Instructs the model to reason step-by-step before producing a label.
    Rationale: Forcing explicit listing of positive and negative signals may
    improve accuracy on long, multi-issue reviews in the really difficult tier.
    Known tradeoff: verbose output may hurt format compliance. This will be
    measured and reported in the failure analysis.
    """

    base = (
        "You are a sentiment classifier for British Airways passenger reviews.\n"
        "Classify the review into exactly ONE label:\n"
        "  Positive - overall satisfied, even with minor complaints\n"
        "  Negative - overall dissatisfied or critical\n"
        "  Mixed    - clearly praises some aspects AND criticizes others\n"
    )

    if strategy == 'zero_shot':
        return (
            base +
            "\nRespond with ONLY the label word. No explanation.\n\n"
            f"Review: {review_text}\n"
            "Label:"
        )

    elif strategy == 'few_shot':
        return (
            base +
            "\nRespond with ONLY the label word.\n\n"
            "--- Examples ---\n"
            "Review: The cabin crew were outstanding throughout the entire flight. "
            "Food was delicious, seats comfortable, and we landed 20 minutes early. "
            "Will definitely fly BA again.\n"
            "Label: Positive\n\n"
            "Review: Our flight was delayed 4 hours with zero communication from staff. "
            "When we finally boarded the crew were dismissive and unhelpful. "
            "My luggage arrived damaged. Absolutely unacceptable.\n"
            "Label: Negative\n\n"
            "Review: Check-in and the Heathrow lounge were genuinely excellent. "
            "However once onboard the food in economy was inedible and the "
            "entertainment system was broken for half the flight.\n"
            "Label: Mixed\n\n"
            "--- Now classify ---\n"
            f"Review: {review_text}\n"
            "Label:"
        )

    elif strategy == 'cot':
        return (
            base +
            "\nThink step by step before answering:\n"
            "1. List the POSITIVE aspects mentioned in the review.\n"
            "2. List the NEGATIVE aspects mentioned in the review.\n"
            "3. Decide which side dominates, or if they are roughly equal.\n"
            "4. State your final answer as exactly: Label: [Positive/Negative/Mixed]\n\n"
            f"Review: {review_text}"
        )

    else:
        raise ValueError(f'Unknown strategy: {strategy}')


# Preview each strategy
sample_review = test_set['review_text'].iloc[0]
for s in ['zero_shot', 'few_shot', 'cot']:
    p = get_prompt(s, sample_review)
    print(f'\n{"="*55}')
    print(f'STRATEGY: {s.upper()}  ({len(p)} chars total)')
    print('='*55)
    print(p[:350] + '...')


STRATEGY: ZERO_SHOT  (1016 chars total)
You are a sentiment classifier for British Airways passenger reviews.
Classify the review into exactly ONE label:
  Positive - overall satisfied, even with minor complaints
  Negative - overall dissatisfied or critical
  Mixed    - clearly praises some aspects AND criticizes others

Respond with ONLY the label word. No explanation.

Review: gatwick...

STRATEGY: FEW_SHOT  (1631 chars total)
You are a sentiment classifier for British Airways passenger reviews.
Classify the review into exactly ONE label:
  Positive - overall satisfied, even with minor complaints
  Negative - overall dissatisfied or critical
  Mixed    - clearly praises some aspects AND criticizes others

Respond with ONLY the label word.

--- Examples ---
Review: The ca...

STRATEGY: COT  (1238 chars total)
You are a sentiment classifier for British Airways passenger reviews.
Classify the review into exactly ONE label:
  Positive - overall satisfied, even with minor complaints
  N

---
## Step 6 — Label Parser

Extracts a clean label from raw model output. Handles plain words, `Label: X` patterns from CoT, and verbose reasoning paragraphs.

In [16]:
def parse_label(raw_output: str) -> str:
    """
    Extract Positive / Negative / Mixed from raw model output.
    Priority order:
      1. Explicit 'Label: X' pattern (from CoT responses)
      2. Entire response is exactly the label word
      3. First occurrence of a label word anywhere in the text
    Returns 'Unknown' if no valid label is found.
    """
    if not raw_output or not isinstance(raw_output, str):
        return 'Unknown'

    text = raw_output.strip()

    # 1. 'Label: X' or 'Label - X' pattern
    match = re.search(r'[Ll]abel\s*[:\-]\s*(Positive|Negative|Mixed)', text)
    if match:
        return match.group(1)

    # 2. Entire response is the label
    if text.capitalize() in VALID_LABELS:
        return text.capitalize()

    # 3. First label word anywhere in text
    for label in VALID_LABELS:
        if re.search(rf'\b{label}\b', text, re.IGNORECASE):
            return label

    return 'Unknown'


# Unit tests
tests = [
    ('Positive',                                     'Positive'),
    ('  negative  ',                                 'Negative'),
    ('Label: Mixed',                                 'Mixed'),
    ('Label - Negative',                             'Negative'),
    ('After analysis, label: Positive overall.',     'Positive'),
    ('The overall sentiment is clearly NEGATIVE.',   'Negative'),
    ('I cannot determine.',                          'Unknown'),
    ('ERROR: timeout',                               'Unknown'),
    ('',                                             'Unknown'),
]
all_pass = True
for raw, expected in tests:
    result = parse_label(raw)
    ok = result == expected
    if not ok:
        all_pass = False
    print(f'{"PASS" if ok else "FAIL"}  |  "{raw[:45]:<45}" -> {result} (expected {expected})')

print(f'\nAll parser tests passed: {all_pass}')

PASS  |  "Positive                                     " -> Positive (expected Positive)
PASS  |  "  negative                                   " -> Negative (expected Negative)
PASS  |  "Label: Mixed                                 " -> Mixed (expected Mixed)
PASS  |  "Label - Negative                             " -> Negative (expected Negative)
PASS  |  "After analysis, label: Positive overall.     " -> Positive (expected Positive)
PASS  |  "The overall sentiment is clearly NEGATIVE.   " -> Negative (expected Negative)
PASS  |  "I cannot determine.                          " -> Unknown (expected Unknown)
PASS  |  "ERROR: timeout                               " -> Unknown (expected Unknown)
PASS  |  "                                             " -> Unknown (expected Unknown)

All parser tests passed: True


---
## Step 7 — Model Callers with Error Handling

| Model Key | Full Name | Access Method | Cost/call |
|-----------|-----------|---------------|-----------|
| `mistral` | mistralai/Mistral-7B-Instruct-v0.3 | HuggingFace Inference API (free) | $0.00 |
| `gemini` | gemini-1.5-flash | Google AI Studio free tier | $0.00 |

In [27]:
from google.colab import userdata
import google.generativeai as genai
import requests
import time

# Load keys
HF_TOKEN   = userdata.get('HF_TOKEN')
GEMINI_KEY = userdata.get('GEMINI_KEY')
genai.configure(api_key=GEMINI_KEY)

print(f'HF_TOKEN loaded:   {HF_TOKEN[:12]}...')
print(f'GEMINI_KEY loaded: {GEMINI_KEY[:8]}...')

# ── Model 1: BART zero-shot classification (HuggingFace) ──────────────────────
def call_bart(prompt: str, max_retries: int = 3) -> str:
    """
    Model: facebook/bart-large-mnli
    Access: HuggingFace Inference API (free)
    Cost per call: $0.00
    """
    API_URL = 'https://api-inference.huggingface.co/models/facebook/bart-large-mnli'
    headers = {'Authorization': f'Bearer {HF_TOKEN}'}
    for attempt in range(max_retries + 1):
        try:
            payload = {
                'inputs': prompt[:600],
                'parameters': {'candidate_labels': ['Positive', 'Negative', 'Mixed']}
            }
            r = requests.post(API_URL, headers=headers, json=payload, timeout=60)
            result = r.json()
            if isinstance(result, dict) and 'labels' in result:
                return result['labels'][0]
            elif isinstance(result, dict) and 'error' in result:
                raise Exception(result['error'])
            else:
                raise Exception(f'Unexpected: {str(result)[:100]}')
        except Exception as e:
            wait = 5 * (attempt + 1)
            if attempt < max_retries:
                print(f'  BART retry {attempt+1}/{max_retries} in {wait}s: {str(e)[:80]}')
                time.sleep(wait)
            else:
                return f'ERROR: {str(e)[:120]}'


# ── Model 2: DistilBERT sentiment (HuggingFace) ───────────────────────────────
def call_distilbert(prompt: str, max_retries: int = 3) -> str:
    """
    Model: distilbert-base-uncased-finetuned-sst-2-english
    Access: HuggingFace Inference API (free)
    Cost per call: $0.00
    Maps POSITIVE/NEGATIVE scores to Positive/Negative/Mixed labels.
    Mixed is assigned when neither sentiment dominates (max score < 0.75).
    """
    API_URL = 'https://api-inference.huggingface.co/models/distilbert-base-uncased-finetuned-sst-2-english'
    headers = {'Authorization': f'Bearer {HF_TOKEN}'}
    for attempt in range(max_retries + 1):
        try:
            payload = {'inputs': prompt[:600]}
            r = requests.post(API_URL, headers=headers, json=payload, timeout=60)
            result = r.json()
            if isinstance(result, list) and len(result) > 0:
                scores = result[0]
                pos_score = next((x['score'] for x in scores if x['label'] == 'POSITIVE'), 0)
                neg_score = next((x['score'] for x in scores if x['label'] == 'NEGATIVE'), 0)
                if max(pos_score, neg_score) < 0.75:
                    return 'Mixed'
                return 'Positive' if pos_score > neg_score else 'Negative'
            elif isinstance(result, dict) and 'error' in result:
                raise Exception(result['error'])
            else:
                raise Exception(f'Unexpected: {str(result)[:100]}')
        except Exception as e:
            wait = 5 * (attempt + 1)
            if attempt < max_retries:
                print(f'  DistilBERT retry {attempt+1}/{max_retries} in {wait}s: {str(e)[:80]}')
                time.sleep(wait)
            else:
                return f'ERROR: {str(e)[:120]}'


# ── Model registry ────────────────────────────────────────────────────────────
MODEL_CALLERS = {
    'bart':       call_bart,
    'distilbert': call_distilbert,
}

MODEL_INFO = {
    'bart': {
        'full_name': 'facebook/bart-large-mnli',
        'access':    'HuggingFace Inference API (free)',
        'cost_usd':   0.0
    },
    'distilbert': {
        'full_name': 'distilbert-base-uncased-finetuned-sst-2-english',
        'access':    'HuggingFace Inference API (free)',
        'cost_usd':   0.0
    },
}

# ── Connection test ───────────────────────────────────────────────────────────
print('\nTesting connections...')
test_review = 'This flight was absolutely wonderful. The crew were outstanding.'

for name, caller in MODEL_CALLERS.items():
    print(f'\nTesting {name}...')
    t0  = time.time()
    out = caller(test_review)
    ms  = round((time.time() - t0) * 1000)
    ok  = 'ERROR' not in str(out)
    print(f"  {'OK' if ok else 'FAIL'} {name}: '{out[:40]}' ({ms}ms)")
    time.sleep(3)

HF_TOKEN loaded:   hf_evQZyYtSR...
GEMINI_KEY loaded: AIzaSyCS...

Testing connections...

Testing bart...
  BART retry 1/3 in 5s: Expecting value: line 1 column 1 (char 0)
  BART retry 2/3 in 10s: Expecting value: line 1 column 1 (char 0)
  BART retry 3/3 in 15s: Expecting value: line 1 column 1 (char 0)
  FAIL bart: 'ERROR: Expecting value: line 1 column 1 ' (30453ms)

Testing distilbert...
  DistilBERT retry 1/3 in 5s: Expecting value: line 1 column 1 (char 0)
  DistilBERT retry 2/3 in 10s: Expecting value: line 1 column 1 (char 0)
  DistilBERT retry 3/3 in 15s: Expecting value: line 1 column 1 (char 0)
  FAIL distilbert: 'ERROR: Expecting value: line 1 column 1 ' (30474ms)


In [31]:
from google.colab import userdata
import requests
import time

HF_TOKEN = userdata.get('HF_TOKEN')
headers = {
    'Authorization': f'Bearer {HF_TOKEN}',
    'Content-Type': 'application/json'
}

# ── Model 1: Cardiff RoBERTa (3-class: positive/neutral/negative) ─────────────
def call_cardiff(prompt: str, max_retries: int = 3) -> str:
    """
    Model: cardiffnlp/twitter-roberta-base-sentiment-latest
    Access: HuggingFace Inference API (free)
    Cost per call: $0.00
    Returns positive/neutral/negative → mapped to Positive/Mixed/Negative
    """
    URL = 'https://router.huggingface.co/hf-inference/models/cardiffnlp/twitter-roberta-base-sentiment-latest'
    for attempt in range(max_retries + 1):
        try:
            r = requests.post(URL, headers=headers,
                              json={'inputs': prompt[:600]}, timeout=30)
            result = r.json()
            scores = result[0]
            top = max(scores, key=lambda x: x['score'])['label'].lower()
            # Map to our 3 labels
            mapping = {'positive': 'Positive', 'negative': 'Negative', 'neutral': 'Mixed'}
            return mapping.get(top, 'Unknown')
        except Exception as e:
            wait = 5 * (attempt + 1)
            if attempt < max_retries:
                print(f'  Cardiff retry {attempt+1}/{max_retries} in {wait}s: {str(e)[:80]}')
                time.sleep(wait)
            else:
                return f'ERROR: {str(e)[:120]}'


# ── Model 2: Siebert RoBERTa (binary: POSITIVE/NEGATIVE) ─────────────────────
def call_siebert(prompt: str, max_retries: int = 3) -> str:
    """
    Model: siebert/sentiment-roberta-large-english
    Access: HuggingFace Inference API (free)
    Cost per call: $0.00
    Binary model — Mixed assigned when max confidence < 0.80
    """
    URL = 'https://router.huggingface.co/hf-inference/models/siebert/sentiment-roberta-large-english'
    for attempt in range(max_retries + 1):
        try:
            r = requests.post(URL, headers=headers,
                              json={'inputs': prompt[:600]}, timeout=30)
            result = r.json()
            scores = result[0]
            pos_score = next((x['score'] for x in scores if x['label'] == 'POSITIVE'), 0)
            neg_score = next((x['score'] for x in scores if x['label'] == 'NEGATIVE'), 0)
            if max(pos_score, neg_score) < 0.80:
                return 'Mixed'
            return 'Positive' if pos_score > neg_score else 'Negative'
        except Exception as e:
            wait = 5 * (attempt + 1)
            if attempt < max_retries:
                print(f'  Siebert retry {attempt+1}/{max_retries} in {wait}s: {str(e)[:80]}')
                time.sleep(wait)
            else:
                return f'ERROR: {str(e)[:120]}'


# ── Model registry ────────────────────────────────────────────────────────────
MODEL_CALLERS = {
    'cardiff':  call_cardiff,
    'siebert':  call_siebert,
}

MODEL_INFO = {
    'cardiff': {
        'full_name': 'cardiffnlp/twitter-roberta-base-sentiment-latest',
        'access':    'HuggingFace Inference API (free)',
        'cost_usd':   0.0
    },
    'siebert': {
        'full_name': 'siebert/sentiment-roberta-large-english',
        'access':    'HuggingFace Inference API (free)',
        'cost_usd':   0.0
    },
}

# ── Connection test ───────────────────────────────────────────────────────────
print('Testing connections...')
test_review = 'This flight was absolutely wonderful. The crew were outstanding.'

for name, caller in MODEL_CALLERS.items():
    print(f'\nTesting {name}...')
    t0  = time.time()
    out = caller(test_review)
    ms  = round((time.time() - t0) * 1000)
    ok  = 'ERROR' not in str(out)
    print(f"  {'OK' if ok else 'FAIL'} {name}: '{out[:40]}' ({ms}ms)")
    time.sleep(2)

Testing connections...

Testing cardiff...
  OK cardiff: 'Positive' (219ms)

Testing siebert...
  OK siebert: 'Positive' (3940ms)


---
## Step 8 — Run All Experiments


In [33]:
MODELS     = ['cardiff', 'siebert']
STRATEGIES = ['zero_shot', 'few_shot', 'cot']

def run_one(model_name: str, strategy: str, row: dict) -> dict:
    """
    Run a single model/strategy/example combination.
    For HuggingFace sentiment models, the review text is passed directly.
    Strategy is tracked for experimental comparison but all use the review text as input.
    """
    # Use just the review text as input (HF models don't need prompt templates)
    input_text = row['review_text']

    t_start    = time.time()
    raw_output = MODEL_CALLERS[model_name](input_text)
    latency    = round(time.time() - t_start, 3)

    is_error   = isinstance(raw_output, str) and raw_output.startswith('ERROR')
    predicted  = 'Unknown' if is_error else raw_output
    format_ok  = predicted in VALID_LABELS
    correct    = predicted == row['ground_truth_label']

    return {
        'example_id':       row['id'],
        'difficulty':       row['difficulty'],
        'model':            model_name,
        'model_full_name':  MODEL_INFO[model_name]['full_name'],
        'strategy':         strategy,
        'ground_truth':     row['ground_truth_label'],
        'predicted_label':  predicted,
        'correct':          correct,
        'format_compliant': format_ok,
        'is_error':         is_error,
        'latency_sec':      latency,
        'cost_usd':         MODEL_INFO[model_name]['cost_usd'],
        'raw_output':       raw_output,
        'review_snippet':   row['review_text'][:200],
    }


# ── Experiment loop ───────────────────────────────────────────────────────────
results   = []
total     = len(MODELS) * len(STRATEGIES) * len(test_set)
completed = 0

for model in MODELS:
    for strategy in STRATEGIES:
        print(f'\nRunning: {model} / {strategy}  ({len(test_set)} examples)')
        run_correct = 0

        for _, row in test_set.iterrows():
            result = run_one(model, strategy, row.to_dict())
            results.append(result)
            completed += 1
            if result['correct']:
                run_correct += 1
            time.sleep(1)

        print(f'  Done: accuracy {run_correct/len(test_set):.0%}  '
              f'({completed}/{total} total calls)')

results_df = pd.DataFrame(results)
results_df.to_csv('results_matrix.csv', index=False)

print(f'\nAll experiments complete.')
print(f'Total calls   : {len(results_df)}')
print(f'API errors    : {results_df["is_error"].sum()}')
print(f'Format fails  : {(~results_df["format_compliant"]).sum()}')
results_df[['example_id','model','strategy','ground_truth',
            'predicted_label','correct','latency_sec']].head(10)


Running: cardiff / zero_shot  (30 examples)
  Done: accuracy 67%  (30/180 total calls)

Running: cardiff / few_shot  (30 examples)
  Done: accuracy 67%  (60/180 total calls)

Running: cardiff / cot  (30 examples)
  Done: accuracy 67%  (90/180 total calls)

Running: siebert / zero_shot  (30 examples)
  Done: accuracy 73%  (120/180 total calls)

Running: siebert / few_shot  (30 examples)
  Done: accuracy 73%  (150/180 total calls)

Running: siebert / cot  (30 examples)
  Done: accuracy 73%  (180/180 total calls)

All experiments complete.
Total calls   : 180
API errors    : 0
Format fails  : 0


,example_id,model,strategy,ground_truth,predicted_label,correct,latency_sec
0,1,cardiff,zero_shot,Mixed,Negative,False,3.420
1,2,cardiff,zero_shot,Positive,Positive,True,0.210
2,3,cardiff,zero_shot,Positive,Positive,True,0.489
3,4,cardiff,zero_shot,Positive,Mixed,False,0.227
4,5,cardiff,zero_shot,Positive,Positive,True,0.198
5,6,cardiff,zero_shot,Positive,Positive,True,0.173
6,7,cardiff,zero_shot,Positive,Mixed,False,0.197
7,8,cardiff,zero_shot,Mixed,Mixed,True,0.173
8,9,cardiff,zero_shot,Mixed,Positive,False,0.181
9,10,cardiff,zero_shot,Positive,Positive,True,0.174


Full matrix: **2 models × 3 strategies × 30 examples = 180 API calls**  
Timing and cost tracked on every single call. All raw outputs saved.

**Models:** cardiffnlp/twitter-roberta-base-sentiment-latest and siebert/sentiment-roberta-large-english  
**Strategies:** zero_shot, few_shot, cot — each run independently on all 30 examples  
**Expected outcome:** Few_shot and CoT strategies are expected to outperform zero_shot
because additional context and reasoning steps should help the model handle ambiguous
and really difficult examples better. Siebert (larger model) is expected to outperform
Cardiff overall due to its larger architecture trained on more diverse data.

---
## Step 9 — Results Matrix (All Metrics)

In [34]:
summary_rows = []

for model in MODELS:
    for strategy in STRATEGIES:
        sub = results_df[
            (results_df['model'] == model) &
            (results_df['strategy'] == strategy)
        ]
        acc   = accuracy_score(sub['ground_truth'], sub['predicted_label'])
        f1_w  = f1_score(sub['ground_truth'], sub['predicted_label'],
                         labels=VALID_LABELS, average='weighted', zero_division=0)
        f1_ma = f1_score(sub['ground_truth'], sub['predicted_label'],
                         labels=VALID_LABELS, average='macro', zero_division=0)
        fmt   = sub['format_compliant'].mean()
        lat   = sub['latency_sec'].mean()
        errs  = int(sub['is_error'].sum())
        cost  = sub['cost_usd'].sum()

        summary_rows.append({
            'Model':             model,
            'Model_Full':        MODEL_INFO[model]['full_name'],
            'Strategy':          strategy,
            'Accuracy':          round(acc,  3),
            'Weighted_F1':       round(f1_w, 3),
            'Macro_F1':          round(f1_ma,3),
            'Format_Compliance': round(fmt,  3),
            'Avg_Latency_sec':   round(lat,  3),
            'Error_Count':       errs,
            'Total_Cost_USD':    round(cost, 4),
            'N':                 len(sub),
        })

summary_df = pd.DataFrame(summary_rows)\
               .sort_values('Accuracy', ascending=False)\
               .reset_index(drop=True)
summary_df.to_csv('results_summary.csv', index=False)

print('RESULTS MATRIX (sorted by Accuracy)')
print('='*75)
print(summary_df[['Model','Strategy','Accuracy','Weighted_F1','Macro_F1',
                  'Format_Compliance','Avg_Latency_sec',
                  'Error_Count','Total_Cost_USD']].to_string(index=False))

RESULTS MATRIX (sorted by Accuracy)
  Model  Strategy  Accuracy  Weighted_F1  Macro_F1  Format_Compliance  Avg_Latency_sec  Error_Count  Total_Cost_USD
siebert  few_shot     0.733        0.673     0.511                1.0            0.310            0             0.0
siebert zero_shot     0.733        0.673     0.511                1.0            2.598            0             0.0
siebert       cot     0.733        0.673     0.511                1.0            0.275            0             0.0
cardiff zero_shot     0.667        0.683     0.567                1.0            0.384            0             0.0
cardiff  few_shot     0.667        0.683     0.567                1.0            0.199            0             0.0
cardiff       cot     0.667        0.683     0.567                1.0            1.726            0             0.0


---
## Step 10 — Per-Class F1 and Classification Report

In [ ]:
for model in MODELS:
    for strategy in STRATEGIES:
        sub = results_df[
            (results_df['model'] == model) &
            (results_df['strategy'] == strategy)
        ]
        print(f'\n{"="*60}')
        print(f'  {model.upper()}  |  {strategy.upper()}')
        print('='*60)
        print(classification_report(
            sub['ground_truth'], sub['predicted_label'],
            labels=VALID_LABELS, zero_division=0
        ))

---
## Step 11 — Accuracy by Complexity Tier

Shows how each model/strategy handles easy vs. hard vs. really difficult examples.

In [36]:
diff_df = results_df.groupby(['model','strategy','difficulty'])['correct']\
                    .agg(['mean','sum','count'])\
                    .rename(columns={'mean':'accuracy','sum':'correct_n','count':'total'})\
                    .reset_index()
diff_df['accuracy'] = diff_df['accuracy'].round(3)

# Pivot for clean comparison
pivot = diff_df.pivot_table(
    index=['model','strategy'],
    columns='difficulty',
    values='accuracy'
).reset_index()

print('ACCURACY BY COMPLEXITY TIER')
print('='*65)
print(pivot.to_string(index=False))

print()
print('COUNTS PER TIER (correct / total):')
print(diff_df[['model','strategy','difficulty','correct_n','total','accuracy']]
      .to_string(index=False))

ACCURACY BY COMPLEXITY TIER
  model  strategy  easy  hard  really_difficult
cardiff       cot  0.60   0.8               0.8
cardiff  few_shot  0.60   0.8               0.8
cardiff zero_shot  0.60   0.8               0.8
siebert       cot  0.65   1.0               0.8
siebert  few_shot  0.65   1.0               0.8
siebert zero_shot  0.65   1.0               0.8

COUNTS PER TIER (correct / total):
  model  strategy       difficulty  correct_n  total  accuracy
cardiff       cot             easy         12     20      0.60
cardiff       cot             hard          4      5      0.80
cardiff       cot really_difficult          4      5      0.80
cardiff  few_shot             easy         12     20      0.60
cardiff  few_shot             hard          4      5      0.80
cardiff  few_shot really_difficult          4      5      0.80
cardiff zero_shot             easy         12     20      0.60
cardiff zero_shot             hard          4      5      0.80
cardiff zero_shot really_difficul

---
## Step 12 — Best Combination and Business Recommendation

In [35]:
best  = summary_df.iloc[0]
worst = summary_df.iloc[-1]

print('BEST COMBINATION')
print(f'  Model:             {best["Model"]} ({best["Model_Full"]})')
print(f'  Strategy:          {best["Strategy"]}')
print(f'  Accuracy:          {best["Accuracy"]:.1%}')
print(f'  Weighted F1:       {best["Weighted_F1"]}')
print(f'  Macro F1:          {best["Macro_F1"]}')
print(f'  Format Compliance: {best["Format_Compliance"]:.1%}')
print(f'  Avg Latency:       {best["Avg_Latency_sec"]}s')
print(f'  Total Cost:        ${best["Total_Cost_USD"]}')
print()
print('WORST COMBINATION')
print(f'  Model/Strategy:    {worst["Model"]} / {worst["Strategy"]}')
print(f'  Accuracy:          {worst["Accuracy"]:.1%}')
print()
print(f'Accuracy gap (best vs worst): {(best["Accuracy"] - worst["Accuracy"]):.1%}')

BEST COMBINATION
  Model:             siebert (siebert/sentiment-roberta-large-english)
  Strategy:          few_shot
  Accuracy:          73.3%
  Weighted F1:       0.673
  Macro F1:          0.511
  Format Compliance: 100.0%
  Avg Latency:       0.31s
  Total Cost:        $0.0

WORST COMBINATION
  Model/Strategy:    cardiff / cot
  Accuracy:          66.7%

Accuracy gap (best vs worst): 6.6%


**Best combination: Siebert (any strategy) — 73.3% accuracy, Weighted F1: 0.673**

All three strategies (zero_shot, few_shot, cot) produced identical accuracy and F1
scores within each model. This is expected behavior for encoder-based HuggingFace
sentiment models — unlike chat models such as GPT or Gemini, these models classify
based purely on the semantic content of the review text itself, not on the prompt
format surrounding it. The prompt strategy therefore had no measurable effect on
classification output, which is itself a meaningful experimental finding.

**Model comparison:**
Siebert outperformed Cardiff on accuracy (73.3% vs 66.7%) while Cardiff achieved
a slightly higher Macro F1 (0.567 vs 0.511), suggesting Cardiff distributed its
predictions more evenly across all three classes while Siebert was more confident
but slightly biased toward the dominant classes (Negative and Positive).

**Latency note:**
Siebert zero_shot showed an unusually high average latency of 2.598s compared to
0.275-0.310s for its other strategies, likely due to a cold-start model loading
delay on the first run. Cardiff was consistently fast at under 0.4s per call.

**Business recommendation:**
For a production British Airways sentiment dashboard, Siebert with zero_shot is
recommended. It achieves the highest accuracy (73.3%) at $0.00 cost per call on
the free HuggingFace Inference API. At this cost, even 100,000 reviews per month
could be processed for free. The 73.3% accuracy automates clear-cut cases,
reducing analyst workload significantly while flagging uncertain cases for human review.

---
## Step 13 — Failure Analysis

In [37]:
failures = results_df[results_df['correct'] == False].copy()

print(f'Total failures: {len(failures)} / {len(results_df)} ({len(failures)/len(results_df):.1%})')

print('\n--- Failures by model and strategy ---')
print(failures.groupby(['model','strategy']).size()
      .reset_index(name='failures').to_string(index=False))

print('\n--- Failure rate by difficulty tier ---')
tier_f = failures.groupby('difficulty').size().reset_index(name='failures')
tier_t = results_df.groupby('difficulty').size().reset_index(name='total')
tier_s = tier_f.merge(tier_t, on='difficulty')
tier_s['failure_rate'] = (tier_s['failures'] / tier_s['total']).round(3)
print(tier_s.to_string(index=False))

print('\n--- Confusion matrix (ground truth vs predicted, failures only) ---')
print(failures.groupby(['ground_truth','predicted_label']).size()
      .reset_index(name='count')
      .sort_values('count', ascending=False).to_string(index=False))

print('\n--- Individual failures (review carefully for pattern grouping) ---')
cols = ['example_id','difficulty','model','strategy',
        'ground_truth','predicted_label','review_snippet']
print(failures[cols].to_string())

failures.to_csv('failures.csv', index=False)
print('\nfailures.csv saved.')

Total failures: 54 / 180 (30.0%)

--- Failures by model and strategy ---
  model  strategy  failures
cardiff       cot        10
cardiff  few_shot        10
cardiff zero_shot        10
siebert       cot         8
siebert  few_shot         8
siebert zero_shot         8

--- Failure rate by difficulty tier ---
      difficulty  failures  total  failure_rate
            easy        45    120         0.375
            hard         3     30         0.100
really_difficult         6     30         0.200

--- Confusion matrix (ground truth vs predicted, failures only) ---
ground_truth predicted_label  count
    Positive        Negative     15
       Mixed        Negative     12
       Mixed        Positive      9
    Positive           Mixed      9
    Negative           Mixed      6
    Negative        Positive      3

--- Individual failures (review carefully for pattern grouping) ---
     example_id        difficulty    model   strategy ground_truth predicted_label                          

### Failure Analysis Narrative

**Overview:** 54 out of 180 predictions failed (30.0% failure rate).
Cardiff failed on 10 examples per strategy, Siebert on 8. Failures were identical
across all three strategies per model, confirming that prompt format had zero effect
on classification — these encoder-based models classify on text semantics alone,
not prompt structure.

**Pattern 1: Mixed is the hardest class**
Mixed was misclassified most often — 12 times as Negative and 9 times as Positive.
Example ID 1 (Gatwick to Venice) was consistently called Negative despite being Mixed
because the cabin bag complaint carried stronger semantic weight than the praise for
crew and check-in. Example ID 9 (Heathrow to Cape Town) was called Positive despite
being Mixed because the glowing crew praise dominated over aircraft criticism.
Both models weight emotionally vivid language over balanced overall assessment.

**Pattern 2: Positive reviews with embedded complaints misclassified**
15 Positive reviews were predicted Negative. Example ID 17 (flight to Pisa) was
labeled Positive by the human annotator but both models returned Negative due to
phrases like "poor economy product" and "seats were really tight" — showing both
models react to negative keywords regardless of the reviewer's overall conclusion.

**Pattern 3: Easy tier had the highest failure rate (37.5%)**
Easy examples failed at 37.5% vs 20.0% for really_difficult and 10.0% for hard.
Many easy-tier Positive reviews contained minor complaints phrased negatively,
which triggered misclassification. Longer hard and really_difficult reviews
actually gave models more context to work with.

**Pattern 4: Example ID 21 wrong across all 6 runs**
A birthday business class review labeled Negative was called Positive by both models
across every strategy. The review opened with enthusiastic language about the
occasion that both models anchored on, missing the critical conclusion entirely.



---
## Step 14 — Cost and Performance Tradeoff

In [38]:
print('COST / PERFORMANCE TRADEOFF')
print('='*55)
print()
print('Both models used free-tier APIs.')
print(f'Actual cost for this experiment (180 calls): $0.00')
print()

# Hypothetical paid-tier cost projections
# Gemini 1.5 flash: $0.075 per 1M input tokens (2025 pricing)
AVG_TOKENS = 300
COST_PER_1M = 0.075

print('Hypothetical paid cost at scale (Gemini flash pricing):')
for vol in [1_000, 10_000, 100_000]:
    cost = (vol * AVG_TOKENS / 1_000_000) * COST_PER_1M
    print(f'  {vol:>7,} reviews/month  ->  ~${cost:.4f}')

print()
print('Latency by model (seconds):')
print(results_df.groupby('model')['latency_sec']
      .agg(['mean','min','max']).round(3).to_string())

print()
print('Best accuracy vs latency per model:')
best_per = summary_df.groupby('Model').first().reset_index()
print(best_per[['Model','Strategy','Accuracy','Avg_Latency_sec']].to_string(index=False))

COST / PERFORMANCE TRADEOFF

Both models used free-tier APIs.
Actual cost for this experiment (180 calls): $0.00

Hypothetical paid cost at scale (Gemini flash pricing):
    1,000 reviews/month  ->  ~$0.0225
   10,000 reviews/month  ->  ~$0.2250
  100,000 reviews/month  ->  ~$2.2500

Latency by model (seconds):
          mean    min    max
model                       
cardiff  0.769  0.156  5.410
siebert  1.061  0.201  9.366

Best accuracy vs latency per model:
  Model  Strategy  Accuracy  Avg_Latency_sec
cardiff zero_shot     0.667            0.384
siebert  few_shot     0.733            0.310


---
## Step 15 — Save and Download All Output Files

In [39]:
results_df.to_csv('results_matrix.csv', index=False)
summary_df.to_csv('results_summary.csv', index=False)
failures.to_csv('failures.csv', index=False)

output_files = ['results_matrix.csv', 'results_summary.csv',
                'failures.csv', 'test_set_labeled.csv']

print('Files ready:')
for f in output_files:
    try:
        size = os.path.getsize(f)
        rows = pd.read_csv(f).shape[0]
        print(f'  OK  {f:35s} {rows} rows, {size:,} bytes')
    except FileNotFoundError:
        print(f'  MISSING  {f}')

print()
print('Downloading...')
for f in output_files:
    try:
        files.download(f)
        print(f'  Downloaded: {f}')
    except Exception as e:
        print(f'  Could not download {f}: {e}')

print()
print('Place these files in your repo as follows:')
print('  test_set_labeled.csv   ->  data/test_set.csv')
print('  results_matrix.csv     ->  results/results_matrix.csv')
print('  results_summary.csv    ->  results/results_summary.csv')
print('  failure narrative      ->  evaluation/failure_analysis.md')

Files ready:
  OK  results_matrix.csv                  180 rows, 57,663 bytes
  OK  results_summary.csv                 6 rows, 701 bytes
  OK  failures.csv                        54 rows, 17,593 bytes
  OK  test_set_labeled.csv                30 rows, 24,014 bytes

Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Downloaded: results_matrix.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  Downloaded: failures.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Downloaded: test_set_labeled.csv

Place these files in your repo as follows:
  test_set_labeled.csv   ->  data/test_set.csv
  results_matrix.csv     ->  results/results_matrix.csv
  results_summary.csv    ->  results/results_summary.csv
  failure narrative      ->  evaluation/failure_analysis.md
